# MiniMax Sparse Attention (Blockwise Top-k)

**Goal**: After this session, you will be able to implement blockwise sparse attention from memory — the mechanism that lets a model attend over million-token contexts at a fraction of the FLOPs.

**Time**: 30 minutes.

## What and Why
Full softmax attention is O(n^2): every query attends to every key. MiniMax Sparse Attention (and the broader Native-Sparse-Attention family) cuts this by (1) scoring *blocks* of keys cheaply, (2) selecting only the **top-k blocks** per query, and (3) running exact attention over just those blocks. At 1M context this is a ~28x reduction in per-token attention compute. The selection is the whole game — exact attention is unchanged, just applied to a sparse subset.

## Key Formula
For a query $q_i$ and key/value sequence split into blocks $B_1, \dots, B_m$ of size $b$:

$$s_{ij} = q_i \cdot \bar{k}_{B_j}, \qquad \bar{k}_{B_j} = \frac{1}{b}\sum_{t \in B_j} k_t$$
$$\mathcal{S}_i = \text{top-}k\big(\{s_{ij}\}_j\big), \qquad
\text{out}_i = \text{softmax}\!\Big(\tfrac{q_i K_{\mathcal{S}_i}^\top}{\sqrt{d}}\Big) V_{\mathcal{S}_i}$$

**Where:**
- $b$ = block size (tokens per block); $m = n/b$ = number of blocks.
- $\bar{k}_{B_j}$ = the block representative: mean of the keys in block $j$.
- $s_{ij}$ = cheap relevance score of block $j$ to query $i$ (the **index branch**).
- $\mathcal{S}_i$ = set of the top-k highest-scoring blocks for query $i$.
- $K_{\mathcal{S}_i}, V_{\mathcal{S}_i}$ = the keys/values from only the selected blocks; exact attention runs here (the **main branch**).

## References
MiniMax Sparse Attention — https://arxiv.org/abs/2606.13392


In [ ]:
import torch
import torch.nn.functional as F
from jaxtyping import Float, Bool
from torch import Tensor

torch.manual_seed(0)

# Single-head setup for clarity. In production these carry (batch, heads, ...) dims;
# the block-selection logic is identical — batching is the Extension.
SEQ, D = 64, 16          # 64 tokens, head_dim 16
BLOCK = 8                # tokens per block  ->  m = 64/8 = 8 blocks
TOP_K = 3                # keep 3 of 8 blocks per query

# Same shapes/role as q,k,v after the per-head projections inside an attention layer.
q = torch.randn(SEQ, D)
k = torch.randn(SEQ, D)
v = torch.randn(SEQ, D)
NUM_BLOCKS = SEQ // BLOCK
print(f"seq={SEQ} d={D} block={BLOCK} num_blocks={NUM_BLOCKS} top_k={TOP_K}")


## Part 1: Score the blocks (the index branch)

Pool each block of keys into one representative vector (mean over the block), then score every block against every query. This is the cheap O(n*m) step that decides *where* the expensive attention will look.

**Predict before you code**: What shape is the score matrix? What does entry `[i, j]` mean?


In [ ]:
predicted_scores_shape = (SEQ, NUM_BLOCKS)
def block_scores(q, k, block):
    num_blocks = k.shape[0] // block
    k_rep = k.view(num_blocks, block, k.shape[-1]).mean(dim=1)   # (num_blocks, d)
    return q @ k_rep.T                                           # (seq, num_blocks)


In [ ]:
# --- Part 1 Validation ---
scores = block_scores(q, k, BLOCK)

print(f"  You predicted scores shape {predicted_scores_shape}; actual {tuple(scores.shape)}"
      f"  (a gap here is the most useful moment of the session)")

expected = (SEQ, NUM_BLOCKS)
assert scores.shape == expected, f"Shape: expected {expected}, got {tuple(scores.shape)}"
print(f"  Shape: {tuple(scores.shape)} -- correct")
print(f"  Range: [{scores.min():.3f}, {scores.max():.3f}]  Mean: {scores.mean():.3f}")

# Reference: explicit block-mean then matmul
k_rep_ref = k.view(NUM_BLOCKS, BLOCK, D).mean(dim=1)   # (num_blocks, d)
ref = q @ k_rep_ref.T                                  # (seq, num_blocks)
max_diff = (scores - ref).abs().max()
try:
    assert torch.allclose(scores, ref, atol=1e-5), f"Max diff {max_diff:.2e}"
except AssertionError:
    print(f"  YOUR : {scores.flatten()[:6].tolist()}")
    print(f"  EXPECT: {ref.flatten()[:6].tolist()}")
    raise
print(f"  Reference match -- correct (max diff {max_diff:.2e})")
print("\nPart 1 complete.")


## Part 2: Top-k block selection -> token-level mask

Each query keeps only its top-k highest-scoring blocks. Turn that selection into a boolean mask over *tokens* (shape `seq x seq`) so Part 3 can apply it to the full attention score matrix.

**Insight**: a per-query top-k over `num_blocks` columns gives you block indices; expanding a block to its `block` member tokens is a `repeat_interleave` along the key axis (or an equivalent reshape-and-broadcast). The selection is per-query, so the mask is *not* symmetric. *In one line, explain why each selected block contributes `block` consecutive True entries to the row.*

**Predict before you code**: How many `True` entries are in each row of the returned mask?


In [ ]:
predicted_true_per_row = TOP_K * BLOCK
def block_selection_mask(scores, top_k, block):
    seq, num_blocks = scores.shape
    top_idx = torch.topk(scores, top_k, dim=-1).indices          # (seq, top_k)
    block_mask = torch.zeros(seq, num_blocks, dtype=torch.bool)
    block_mask.scatter_(1, top_idx, True)                        # (seq, num_blocks)
    return block_mask.repeat_interleave(block, dim=1)            # (seq, seq)


In [ ]:
# --- Part 2 Validation ---
scores = block_scores(q, k, BLOCK)
mask = block_selection_mask(scores, TOP_K, BLOCK)

print(f"  You predicted {predicted_true_per_row} True per row; actual {int(mask[0].sum())}")

assert mask.shape == (SEQ, SEQ), f"Shape: expected {(SEQ, SEQ)}, got {tuple(mask.shape)}"
assert mask.dtype == torch.bool, f"dtype: expected bool, got {mask.dtype}"
print(f"  Shape: {tuple(mask.shape)} dtype {mask.dtype} -- correct")

# Each query keeps exactly top_k blocks -> top_k * block tokens.
per_row = mask.sum(dim=1)
assert (per_row == TOP_K * BLOCK).all(), \
    f"Each row must select top_k*block={TOP_K*BLOCK} tokens; got counts {per_row.unique().tolist()}"
print(f"  Each query attends to exactly {TOP_K*BLOCK} tokens -- correct")

# The selected blocks must be the argmax-by-score ones (check query 0).
top_blocks_0 = torch.topk(scores[0], TOP_K).indices.sort().values
sel_blocks_0 = mask[0].view(NUM_BLOCKS, BLOCK).any(dim=1).nonzero().flatten().sort().values
assert torch.equal(top_blocks_0, sel_blocks_0), \
    f"Selected blocks {sel_blocks_0.tolist()} != top-k blocks {top_blocks_0.tolist()}"
print(f"  Query 0 selected blocks {sel_blocks_0.tolist()} = its top-{TOP_K} -- correct")
print("\nPart 2 complete.")


## Part 3: Sparse attention (the main branch)

Run exact scaled-dot-product attention, but mask out every key token that wasn't selected. The payoff check: when `top_k == num_blocks` (nothing pruned), this must reproduce *full* attention exactly.

**Predict before you code**: With `top_k == num_blocks`, what should `max|sparse - full|` be?


In [ ]:
predicted_max_diff_full = 1e-6
def sparse_attention(q, k, v, block, top_k):
    d = q.shape[-1]
    scores = block_scores(q, k, block)
    mask = block_selection_mask(scores, top_k, block)            # (seq, seq) bool
    attn = (q @ k.T) / (d ** 0.5)
    attn = attn.masked_fill(~mask, float("-inf"))
    return torch.softmax(attn, dim=-1) @ v


In [ ]:
# --- Part 3 Validation ---
out = sparse_attention(q, k, v, BLOCK, TOP_K)

assert out.shape == (SEQ, D), f"Shape: expected {(SEQ, D)}, got {tuple(out.shape)}"
print(f"  Shape: {tuple(out.shape)} -- correct")
print(f"  Range: [{out.min():.3f}, {out.max():.3f}]  Mean: {out.mean():.3f}")

# Invariant: with no pruning, sparse == full attention.
full_out = sparse_attention(q, k, v, BLOCK, NUM_BLOCKS)
ref_full = F.scaled_dot_product_attention(q[None], k[None], v[None])[0]
max_diff = (full_out - ref_full).abs().max()
print(f"  You predicted max|sparse-full| ~{predicted_max_diff_full}; actual {max_diff:.2e}")
try:
    assert torch.allclose(full_out, ref_full, atol=1e-5), f"Max diff {max_diff:.2e}"
except AssertionError:
    print(f"  YOUR : {full_out.flatten()[:6].tolist()}")
    print(f"  EXPECT: {ref_full.flatten()[:6].tolist()}")
    raise
print(f"  top_k==num_blocks reproduces full attention -- correct (max diff {max_diff:.2e})")

# Sparsity actually changed the answer (otherwise selection is a no-op).
assert not torch.allclose(out, ref_full, atol=1e-3), \
    "Sparse (top_k=3) output equals full output -- selection isn't pruning anything"
keep = TOP_K / NUM_BLOCKS
print(f"  Sparse output differs from full (good) -- attending {keep:.0%} of keys -> ~{1/keep:.1f}x less attn compute")
print("\nPart 3 complete.")


## Session Debrief

Write your answers in the code cell below (typing is overt retrieval — far stronger than answering in your head). Don't scroll up.
1. What are the two branches of MiniMax Sparse Attention, and what does each compute?
2. How is a per-query top-k *block* selection turned into a token-level attention mask?
3. Why must `top_k == num_blocks` reproduce full attention exactly?

**Check yourself**: the worked solution is in `_solutions/` — open it (and the paper) to grade. Opening it ends the retrieval rep, so answer first.

**Challenge**: close this notebook, open a blank one, and rewrite Part 3 (`sparse_attention`) from scratch without looking back.


In [ ]:
debrief = """
1.
2.
3.
"""  # type your recall here before checking _solutions/


In [ ]:
# --- Session log: fill the `___` then run (appends one line to .practice-log.jsonl) ---
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "llm/19-MiniMax-Sparse-Attention",
    "date": datetime.date.today().isoformat(),
    "budget_min": 30,
    "actual_min": ___,
    "parts": [
        {"n": 1, "tier": 2, "outcome": "___"},   # unaided | hint | solution | failed
        {"n": 2, "tier": 2, "outcome": "___"},
        {"n": 3, "tier": 2, "outcome": "___"},
    ],
    "difficulty": ___,                            # 1 (trivial) .. 5 (over my head)
    "stuck": "___",
}
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")


## Extension (Optional)
- **Causal version**: add a causal mask so query `i` can only select/attend blocks ending at or before `i`. (This is what real decoding uses.)
- **Batch + heads**: lift every function to `(batch, heads, seq, d)`. Where does `repeat_interleave` / the top-k `dim` change?
- **Break it**: select the *bottom*-k blocks instead of top-k. Does the output still look reasonable? Why is that the wrong choice?


## Anki Cards

**Card 1**
Front: You need attention over a 1M-token context but O(n^2) is too expensive. What does the "index branch" of sparse attention compute?
Back: Cheap per-block scores -> top-k blocks per query

**Card 2**
Front: Sparse-attention block scoring — what's the block's representative vector?
Back: Mean of the block's keys

**Card 3**
Front: Your blockwise sparse attention with top_k = num_blocks doesn't match full attention. What's broken?
Back: Masking/softmax (no-prune case must be identity)
